In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
# Q1
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)


72

In [3]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)


In [4]:
#Q2
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict= {'filename': 3.0, 'content': 0.5}, #{"question": 2.0, "section": 0.5},
    #filter_dict={"course": "llm-zoomcamp"},
    num_results=1
)
search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [5]:
# Q3

from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

from gitsource import chunk_documents


documents = load_faq_data()
chunks = chunk_documents(documents, size=2000, step=1000)
index = build_index(chunks)

openai_client = OpenAI(
    base_url="http://localhost:11434/v1/", 
    api_key='ollama'
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

ResponseUsage(input_tokens=2692, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=632, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=3324)
The agentic loop continues calling the model as long as the model suggests using tools or making function calls.

The core mechanism is a `while` loop that checks for tool usage after each API call. The process keeps running until the model returns an answer without any required function calls, at which point the loop breaks.

In summary:
*   **Continuation:** If the model's response contains an item of type `"function_call"`, a flag (`has_function_calls`) is set to `True`, causing the loop to run again with the tool's result included in the next message history.
*   **Termination:** The loop stops when `has_function_calls` remains `False` after processing a model response, indicating that the model has provided a final answer and requires no further tool use.


In [7]:
INSTRUCTIONS = """

You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
"""

In [6]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""


In [7]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append('Filename: ' + doc['filename'])
        lines.append('Content: ' + doc['content'])
        lines.append("")

    return "\n".join(lines).strip()

In [8]:
def build_prompt(search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [9]:
prompt = build_prompt(search_results)

print(prompt)


Question:
How does the agentic loop keep calling the model until it stops?

Context:
Filename: 01-agentic-rag/lessons/14-agentic-loop.md
Content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The b

In [39]:
#4
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [10]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [19]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [14]:
messages = [
    {"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}
]

response = openai_client.responses.create(
    model="gemma4",
    input=messages,
)

response.output_text

'This is a sophisticated comparison that addresses the frontier of current AI application architecture.\n\nIn short: **RAG** is an improvement over simple LLM knowledge limitations by giving it information. The **Agentic Loop** is an advancement over RAG because it gives the system the ability to *plan, decide, and execute* multi-step actions in the real world, rather than just reading provided text.\n\nHere is a detailed breakdown of how the agentic loop works and how it fundamentally differs from plain RAG.\n\n---\n\n## 🧠 Part 1: How Does Plain RAG Work? (The Reader)\n\nRetrieval-Augmented Generation (RAG) is primarily an **information retrieval** technique used to ground a large language model\'s response in factual, external data, thereby reducing hallucinations and limiting the scope of knowledge.\n\n### The Mechanism\nRAG is fundamentally a **linear, single-pass process:**\n\n1.  **Query:** The user submits a question or prompt ($Q$).\n2.  **Retrieval (The Search):** The system u

In [33]:
    def search(self, query, num_results=1):
        boost_dict = {'filename': 3.0, 'content': 0.5} #{'question': 3.0, 'section': 0.5}
        #filter_dict = {'filename': self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict #,
            #filter_dict=filter_dict
        )

In [34]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "How does the agentic loop work, and how is it different from plain RAG?"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [35]:
# Q6

response = openai_client.responses.create(
    model="gemma4",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseReasoningItem(id='rs_resp_19697', summary=[Summary(text='The user is asking a specific technical question about AI concepts: "How does the agentic loop work, and how is it different from plain RAG?".\n\nI have a tool `search` with a function that takes a `query`. The description of this function\'s parameter `query` is exactly the question the user has asked: "How does the agentic loop work, and how is it different from plain RAG?".\n\nTherefore, I should use the `search` tool to find an answer to this precise query.', type='summary_text')], type='reasoning', content=None, encrypted_content='The user is asking a specific technical question about AI concepts: "How does the agentic loop work, and how is it different from plain RAG?".\n\nI have a tool `search` with a function that takes a `query`. The description of this function\'s parameter `query` is exactly the question the user has asked: "How does the agentic loop work, and how is it different from plain RAG?".\n\nTherefore

In [36]:
import json

call = response.output[0]

# args = json.loads(call.arguments)

# results = search(**args)
# result_json = json.dumps(results, indent=2)


# Assuming 'response' is the object or list of items returned by Ollama/Gemma4

for item in response.output: # Or whatever property holds your response items
    
    # Check if the item is a function/tool call (which HAS arguments)
    if hasattr(item, 'arguments'):
        print(f"Tool Name: {item.name}")
        print(f"Arguments: {item.arguments}")
        
    # Check if the item is just a reasoning/thinking block
    elif item.type == "reasoning": 
        print(f"Reasoning output: {item.content}")
        
    # Check if it's normal text output
    elif item.type == "message":
        print(f"Message output: {item.content}")


Reasoning output: None
Tool Name: search
Arguments: {"query":"How does the agentic loop work, and how is it different from plain RAG?"}


In [37]:
## Wrap in function 
def agent_loop(instructions, question, model="gemma4") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [38]:
agent_loop(instructions, "How does the agentic loop work, and how is it different from plain RAG?")

iteration #1...
function_call: search {"query":"How does the agentic loop work and how is it different from plain RAG"}


TypeError: search() missing 1 required positional argument: 'self'